# APX (Approximate Next-Token Probability) — PersianLLaMA

Computes APX bias scores for PersianLLaMA on Persian-translated StereoSet sentences.

In [ ]:
!pip install peft accelerate bitsandbytes transformers -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import torch, math
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
!huggingface-cli login

In [ ]:
df = pd.read_excel('/content/drive/MyDrive/NewStreoSet_Dataset.xlsx')
print(df.shape)

In [ ]:
MODEL_NAME = 'Narabzad/PersianLLaMA-13B'
tokenizer_fa = AutoTokenizer.from_pretrained(MODEL_NAME)
model_fa = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map='auto')
model_fa.eval()

In [ ]:
def compute_ppl(sentence, tokenizer, model, device='cuda'):
 inputs = tokenizer(sentence, return_tensors='pt').to(device)
 with torch.no_grad():
 loss = model(**inputs, labels=inputs['input_ids']).loss
 return math.exp(loss.item())

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
ppl_fa = []
for _, row in tqdm(df.iterrows(), total=len(df)):
 ppl_fa.append(compute_ppl(str(row['Persian']), tokenizer_fa, model_fa, device))
df['PPL_Persian'] = ppl_fa

In [ ]:
mean_ppl_fa = df.groupby('Persian_Target')['PPL_Persian'].mean().to_dict()
df['MeanTargetPPL_Persian'] = df['Persian_Target'].map(mean_ppl_fa)
df['APX_Persian'] = df['PPL_Persian'] / df['MeanTargetPPL_Persian'] * 100

In [ ]:
print('--- APX Bias by Type (PersianLLaMA) ---')
for btype, group in df.groupby('bias_type'):
 stereo = group[group['label_name']=='stereotype']['APX_Persian'].mean()
 anti = group[group['label_name']=='anti-stereotype']['APX_Persian'].mean()
 print(f'{btype}: stereo={stereo:.2f}, anti={anti:.2f}, diff={stereo-anti:.2f}')

In [ ]:
df.to_csv('/content/drive/MyDrive/APX_persianLLaMA.csv')
print('Saved')